# 01 · Ingesta y Limpieza de Datos
**Proyecto:** IW Resource Management – Caso Familia Miranda  
**Autor:** Diego Ballesteros  
**Fecha:** 2025  
**Notebook:** 1 de 4

---

## Objetivo
Cargar, inspeccionar y limpiar todos los archivos fuente del caso de negocio:
- `presupuesto.xlsx` — hojas Agosto 2023 y Septiembre 2023
- `Gastos_Papa_202308.txt` y `Gastos_Papa_202309.txt` — CSV separado por `;`
- `Gastos_Mama_202308.txt` y `Gastos_Mama_202309.txt` — CSV separado por `;`
- `Gastos_Hijo_202309.xlsx` — solo disponible para septiembre

Al finalizar este notebook se exportan los datos limpios a `data/processed/` listos para ser cargados al modelo relacional en el notebook 02.

---

## Estructura del notebook
1. Setup e importaciones
2. Carga de archivos fuente
3. Inspección y diagnóstico de calidad
4. Limpieza y estandarización
5. Mapeo de categorías
6. Exportación de datos procesados

---
## 1. Setup e importaciones

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Rutas del proyecto ──────────────────────────────────────────────────────
BASE_DIR   = Path('..') 
RAW_DIR    = BASE_DIR / 'data' / 'raw'
PROC_DIR   = BASE_DIR / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuración de display ────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✅ Setup completado')
print(f'   RAW_DIR  → {RAW_DIR.resolve()}')
print(f'   PROC_DIR → {PROC_DIR.resolve()}')

✅ Setup completado
   RAW_DIR  → /home/diego/iw-proyecto/data/raw
   PROC_DIR → /home/diego/iw-proyecto/data/processed


---
## 2. Carga de archivos fuente

### 2.1 Archivos de gastos CSV (Papá y Mamá)

Los archivos `.txt` usan como separador `;` y cada valor viene envuelto en comillas simples `'`. Esto requiere un parseo específico — no es un CSV estándar.

In [2]:
def cargar_gasto_txt(ruta: Path, miembro: str, mes: str) -> pd.DataFrame:
    """
    Carga un archivo de gastos en formato TXT con separador ';'
    y valores envueltos en comillas simples.

    Parámetros
    ----------
    ruta    : Path al archivo fuente
    miembro : Nombre del miembro de la familia ('papa', 'mama')
    mes     : Mes en formato 'YYYY-MM' para trazabilidad

    Retorna
    -------
    DataFrame con columna adicional 'miembro' y 'mes_origen'
    """
    df = pd.read_csv(
        ruta,
        sep=';',
        quotechar="'",
        encoding='utf-8'
    )
    df.columns = df.columns.str.strip().str.replace("'", '', regex=False)
    df['miembro']    = miembro
    df['mes_origen'] = mes
    return df


# Cargar los cuatro archivos TXT
df_papa_08  = cargar_gasto_txt(RAW_DIR / 'Gastos_Papa_202308.txt', 'papa',  '2023-08')
df_papa_09  = cargar_gasto_txt(RAW_DIR / 'Gastos_Papa_202309.txt', 'papa',  '2023-09')
df_mama_08  = cargar_gasto_txt(RAW_DIR / 'Gastos_Mama_202308.txt', 'mama',  '2023-08')
df_mama_09  = cargar_gasto_txt(RAW_DIR / 'Gastos_Mama_202309.txt', 'mama',  '2023-09')

print(f'Papa  Ago: {len(df_papa_08):>3} registros')
print(f'Papa  Sep: {len(df_papa_09):>3} registros')
print(f'Mamá  Ago: {len(df_mama_08):>3} registros')
print(f'Mamá  Sep: {len(df_mama_09):>3} registros')
print()
df_papa_09.head(3)

Papa  Ago: 117 registros
Papa  Sep: 126 registros
Mamá  Ago:  27 registros
Mamá  Sep:  46 registros



,fecha,flujo casa mes,valor,Forma de Pago,idCategoria,miembro,mes_origen
0,2023-09-30,Pago Legon,"85,000.00",Debito,PAGO INTERNET,papa,2023-09
1,2023-09-30,Pago gas,"8,736.00",Debito,PAGO GAS,papa,2023-09
2,2023-09-30,Pago Agua,"42,630.00",Debito,PAGO AGUA,papa,2023-09


### 2.2 Archivos de gastos XLSX (Hijo)

In [3]:
df_hijo_09 = pd.read_excel(
    RAW_DIR / 'Gastos_Hijo_202309.xlsx',
    sheet_name='Sheet1'
)
df_hijo_09['miembro']    = 'hijo'
df_hijo_09['mes_origen'] = '2023-09'

print(f'Hijo  Sep: {len(df_hijo_09):>3} registros')
df_hijo_09.head(3)

Hijo  Sep:  56 registros


,fecha,flujo casa mes,valor,Forma de Pago,idCategoria,miembro,mes_origen
0,2023-09-01,contrato hijo,2500000,efectivo,contrato hijo,hijo,2023-09
1,2023-09-01,helados casa,3000,efectivo,COMIDAS AFUERA,hijo,2023-09
2,2023-09-01,cafe americano,5000,efectivo,COMIDAS AFUERA,hijo,2023-09


### 2.3 Presupuesto (ambos meses)

In [4]:
# Filas que no son rubros reales (totales, secciones, encabezados)
FILAS_EXCLUIR = {
    'Ingresos', 'Egresos Colombia', 'Varios:', 'TOTAL INGRESOS',
    'TOTAL EGRESOS', 'Ahorro Esperado', 'total ahorros'
}

def cargar_presupuesto(ruta: Path, hoja: str, mes: str) -> pd.DataFrame:
    """
    Carga una hoja del presupuesto y retorna solo las filas
    que representan rubros reales con su valor planeado.
    Las fórmulas de Excel se leen como NaN; se preserva el valor_planeado.
    """
    df = pd.read_excel(ruta, sheet_name=hoja, usecols='A:C',
                       header=0, engine='openpyxl')
    df.columns = ['rubro', 'valor_planeado', 'valor_real']
    # Filtrar filas válidas
    df = df[
        df['rubro'].notna() &
        ~df['rubro'].isin(FILAS_EXCLUIR) &
        df['valor_planeado'].apply(lambda x: isinstance(x, (int, float)))
    ].copy()
    df['mes'] = mes
    df = df[['mes', 'rubro', 'valor_planeado']].reset_index(drop=True)
    return df


df_ppto_08 = cargar_presupuesto(RAW_DIR / 'presupuesto.xlsx', 'Agosto 2023',     '2023-08')
df_ppto_09 = cargar_presupuesto(RAW_DIR / 'presupuesto.xlsx', 'Septiembre 2023', '2023-09')

print(f'Presupuesto Agosto:     {len(df_ppto_08)} rubros')
print(f'Presupuesto Septiembre: {len(df_ppto_09)} rubros')
df_ppto_09.head(8)

Presupuesto Agosto:     37 rubros
Presupuesto Septiembre: 37 rubros


,mes,rubro,valor_planeado
0,2023-09,Contrato papa,"10,574,000.00"
1,2023-09,Contrato mama,"8,500,000.00"
2,2023-09,contrato hijo,"2,500,000.00"
3,2023-09,CDT compra carro,"10,000,000.00"
4,2023-09,CUOTA CASA,"3,700,000.00"
5,2023-09,CUOTA NETFLIX,"15,000.00"
6,2023-09,PAGO ADMINISTACION CASA,"122,000.00"
7,2023-09,PAGO COOPERATIVA,"100,287.00"


---
## 3. Inspección y diagnóstico de calidad

Antes de limpiar, es fundamental entender exactamente qué problemas tienen los datos. Un buen analista **documenta** los hallazgos antes de corregirlos.

In [5]:
# Consolidar todos los gastos en un solo DataFrame para inspección
df_gastos_raw = pd.concat(
    [df_papa_08, df_papa_09, df_mama_08, df_mama_09, df_hijo_09],
    ignore_index=True
)

print('═' * 55)
print('DIAGNÓSTICO DE CALIDAD — GASTOS CONSOLIDADOS')
print('═' * 55)
print(f'Total registros        : {len(df_gastos_raw)}')
print(f'Columnas               : {list(df_gastos_raw.columns)}')
print()
print('── Valores nulos por columna ──')
print(df_gastos_raw.isnull().sum())
print()
print('── Tipos de datos ──')
print(df_gastos_raw.dtypes)

═══════════════════════════════════════════════════════
DIAGNÓSTICO DE CALIDAD — GASTOS CONSOLIDADOS
═══════════════════════════════════════════════════════
Total registros        : 372
Columnas               : ['fecha', 'flujo casa mes', 'valor', 'Forma de Pago', 'idCategoria', 'miembro', 'mes_origen']

── Valores nulos por columna ──
fecha             0
flujo casa mes    0
valor             0
Forma de Pago     0
idCategoria       0
miembro           0
mes_origen        0
dtype: int64

── Tipos de datos ──
fecha             object
flujo casa mes       str
valor             object
Forma de Pago        str
idCategoria          str
miembro              str
mes_origen           str
dtype: object


In [6]:
# ── Detectar valores no numéricos en la columna 'valor' ─────────────────────
# Este es un problema conocido: Papa Agosto tiene '8MIL PESOS'

valores_invalidos = df_gastos_raw[
    pd.to_numeric(df_gastos_raw['valor'], errors='coerce').isna() &
    df_gastos_raw['valor'].notna()
]

if len(valores_invalidos) > 0:
    print('⚠️  VALORES NO NUMÉRICOS DETECTADOS:')
    print(valores_invalidos[['fecha', 'flujo casa mes', 'valor', 'miembro', 'mes_origen']])
else:
    print('✅ No se encontraron valores no numéricos')

⚠️  VALORES NO NUMÉRICOS DETECTADOS:
         fecha  flujo casa mes       valor miembro mes_origen
58  2023-08-12  Mercado Origen  8MIL PESOS    papa    2023-08


In [7]:
# ── Detectar filas de encabezado repetidas (artefactos del parseo) ───────────
filas_encabezado = df_gastos_raw[
    df_gastos_raw['fecha'].astype(str).str.contains('fecha|idCategoria', case=False, na=False)
]
print(f'Filas de encabezado repetidas: {len(filas_encabezado)}')
if len(filas_encabezado) > 0:
    print(filas_encabezado)

# ── Distribución por miembro ─────────────────────────────────────────────────
print()
print('── Registros por miembro y mes ──')
print(df_gastos_raw.groupby(['miembro', 'mes_origen']).size().to_string())

Filas de encabezado repetidas: 0

── Registros por miembro y mes ──
miembro  mes_origen
hijo     2023-09        56
mama     2023-08        27
         2023-09        46
papa     2023-08       117
         2023-09       126


---
## 4. Limpieza y estandarización

Con base en el diagnóstico, se aplican las siguientes correcciones:

| # | Problema | Acción |
|---|----------|--------|
| 1 | Valor `'8MIL PESOS'` no numérico | Imputar `8000` y registrar en log |
| 2 | Filas de encabezado duplicadas en CSV | Eliminar |
| 3 | Columna `fecha` como string | Convertir a `datetime` |
| 4 | Espacios y mayúsculas inconsistentes en categorías | Normalizar |
| 5 | Filas `idCategoria == 'efectivo'` (error de captura) | Eliminar |

In [8]:
# Log de decisiones de limpieza — para documentar en la entrega
limpieza_log = []

df_gastos = df_gastos_raw.copy()

# ── 1. Eliminar filas de encabezado repetidas ────────────────────────────────
mask_header = df_gastos['fecha'].astype(str).str.contains(
    'fecha|idCategoria', case=False, na=False
)
n_header = mask_header.sum()
df_gastos = df_gastos[~mask_header].copy()
limpieza_log.append(f'[OK] Eliminadas {n_header} fila(s) de encabezado repetidas')

# ── 2. Eliminar filas con idCategoria == 'efectivo' (error de captura) ───────
mask_efectivo = df_gastos['idCategoria'].astype(str).str.strip().str.lower() == 'efectivo'
n_efectivo = mask_efectivo.sum()
df_gastos = df_gastos[~mask_efectivo].copy()
limpieza_log.append(f'[OK] Eliminadas {n_efectivo} fila(s) con idCategoria="efectivo" (error de captura)')

# ── 3. Imputar valor '8MIL PESOS' ────────────────────────────────────────────
mask_8mil = df_gastos['valor'].astype(str).str.upper().str.contains('8MIL', na=False)
n_8mil = mask_8mil.sum()
df_gastos.loc[mask_8mil, 'valor'] = 8000
limpieza_log.append(
    f'[IMPUTADO] {n_8mil} registro(s) con valor "8MIL PESOS" → 8000 '
    f'(Papá, Ago 2023, Mercado Origen)'
)

# ── 4. Convertir tipos ────────────────────────────────────────────────────────
df_gastos['fecha'] = pd.to_datetime(df_gastos['fecha'], errors='coerce')
df_gastos['valor'] = pd.to_numeric(df_gastos['valor'], errors='coerce')

# ── 5. Limpiar strings ────────────────────────────────────────────────────────
for col in ['flujo casa mes', 'Forma de Pago', 'idCategoria']:
    df_gastos[col] = df_gastos[col].astype(str).str.strip()

# ── 6. Renombrar columnas a snake_case ────────────────────────────────────────
df_gastos = df_gastos.rename(columns={
    'flujo casa mes' : 'descripcion',
    'Forma de Pago'  : 'forma_pago',
    'idCategoria'    : 'id_categoria'
})

# ── Resumen ───────────────────────────────────────────────────────────────────
print('RESUMEN DE LIMPIEZA')
print('─' * 50)
for entry in limpieza_log:
    print(' ', entry)
print()
print(f'Registros antes : {len(df_gastos_raw)}')
print(f'Registros después: {len(df_gastos)}')
print()
df_gastos.dtypes

RESUMEN DE LIMPIEZA
──────────────────────────────────────────────────
  [OK] Eliminadas 0 fila(s) de encabezado repetidas
  [OK] Eliminadas 0 fila(s) con idCategoria="efectivo" (error de captura)
  [IMPUTADO] 1 registro(s) con valor "8MIL PESOS" → 8000 (Papá, Ago 2023, Mercado Origen)

Registros antes : 372
Registros después: 372



fecha           datetime64[us]
descripcion                str
valor                  float64
forma_pago                 str
id_categoria               str
miembro                    str
mes_origen                 str
dtype: object

---
## 5. Mapeo de categorías

Las categorías de los archivos de gastos **no coinciden exactamente** con los rubros del presupuesto. Se construye una tabla de equivalencias para poder cruzar ambas fuentes.

Categorías en gastos que **no tienen rubro en el presupuesto** son un hallazgo analítico valioso — responden la pregunta: *¿qué gastos no están en rubros presupuestados?*

In [9]:
# ── Tabla de mapeo: categoria_gasto → rubro_presupuesto ──────────────────────
# None = la categoría no tiene rubro equivalente en el presupuesto

MAPEO_CATEGORIAS = {
    # Ingresos
    'Contrato papa'                   : 'Contrato papa',
    'Contrato mama'                   : 'Contrato mama',
    'contrato hijo'                   : 'contrato hijo',

    # Vivienda y servicios
    'CUOTA CASA'                      : 'CUOTA  CASA',
    'PAGO ADMINISTACION CASA'         : 'PAGO ADMINISTACION CASA',
    'PAGO LUZ'                        : 'PAGO LUZ',
    'PAGO AGUA'                       : 'PAGO AGUA',
    'PAGO GAS'                        : 'PAGO GAS',
    'PAGO CLARO MOVIL'                : 'PAGO CLARO MOVIL',
    'PAGO INTERNET'                   : 'PAGO INTERNET',
    'COSAS DE CASA'                   : 'COSAS DE CASA',
    'JARDIN'                          : 'JARDIN',

    # Salud
    'PAGO SALUD Y PENSIONES'          : 'PAGO SALUD Y PENSIONES',
    'PAGO MEDICINA PREPAGADA'         : 'PAGO MEDICINA PREPAGADA',
    'MEDICINA - MEDICO'               : 'CITAS MEDICAS',

    # Alimentación
    'MERCADO'                         : 'MERCADO',
    'COMIDAS AFUERA'                  : 'COMIDAS AFUERA',
    'SANCOCHO'                        : 'COMIDAS AFUERA',   # reclasificado

    # Transporte
    'TRANSPORTE CARRO / MOTO'         : 'TRANSPORTE CARRO/MOTO/GASOLINA',
    'TRANSPORTE TAXI'                 : 'TRANSPORTE TAXI',
    'VIAJES'                          : 'VIAJES',

    # Finanzas y pagos
    'PAGO TARJETA NUBE'               : 'PAGO TARJETA NUBE',
    'CUOTAS DE MANEJO'                : None,   # sin rubro → gasto no presupuestado
    'PRESTAMO'                        : None,   # sin rubro → gasto no presupuestado
    'RETIRO CAJERO'                   : None,   # sin rubro → gasto no presupuestado
    'inversiones'                     : None,   # sin rubro → gasto no presupuestado

    # Familia y religión
    'OFRENDAS FAMILIA'                : 'AYUDAS A FAMILIARES',
    'DIEZMO'                          : 'DIEZMO',

    # Ocio y deporte
    'DEPORTE ENTRENO + INSCRIPCIONES' : 'DEPORTE ENTRENO + INSCRIPCIONES',
    'ENTRETENIMIENTO/VIAJE'           : 'ENTRENAMIENTO/VIAJE',

    # Otros
    'PELUQUERIA'                      : 'PELUQUERIA',
    'ROPA'                            : 'ROPA',
    'REGALOS'                         : 'REGALOS',
    'LIBROS'                          : 'LIBROS',
    'OTROS'                           : 'OTROS',
}

# Aplicar mapeo
df_gastos['rubro_presupuesto'] = df_gastos['id_categoria'].map(MAPEO_CATEGORIAS)

# Verificar categorías sin mapeo
sin_mapeo = df_gastos[df_gastos['rubro_presupuesto'].isna() & 
                       ~df_gastos['id_categoria'].isin(['CUOTAS DE MANEJO','PRESTAMO',
                                                         'RETIRO CAJERO','inversiones'])]

if len(sin_mapeo) > 0:
    print('⚠️  Categorías sin mapeo definido:')
    print(sin_mapeo['id_categoria'].value_counts())
else:
    print('✅ Todas las categorías tienen mapeo definido')

print()
print('── Gastos NO presupuestados (sin rubro) ──')
print(
    df_gastos[df_gastos['rubro_presupuesto'].isna()]
    .groupby('id_categoria')['valor']
    .agg(['count', 'sum'])
    .rename(columns={'count':'# registros', 'sum':'total_gasto'})
    .sort_values('total_gasto', ascending=False)
)

✅ Todas las categorías tienen mapeo definido

── Gastos NO presupuestados (sin rubro) ──
                  # registros   total_gasto
id_categoria                               
inversiones                 4 78,314,864.00
PRESTAMO                    2  4,000,000.00
RETIRO CAJERO               4  2,000,000.00
CUOTAS DE MANEJO            5    211,431.81


---
## 6. Exportación de datos procesados

Se exportan tres archivos limpios a `data/processed/` que serán la entrada del notebook 02 (modelo relacional).

In [10]:
# ── Gastos consolidados (todos los miembros, ambos meses) ────────────────────
df_gastos_final = df_gastos[[
    'fecha', 'descripcion', 'valor', 'forma_pago',
    'id_categoria', 'rubro_presupuesto', 'miembro', 'mes_origen'
]].copy()

ruta_gastos = PROC_DIR / 'gastos_clean.csv'
df_gastos_final.to_csv(ruta_gastos, index=False)
print(f'✅ gastos_clean.csv  → {len(df_gastos_final)} registros')

# ── Presupuesto consolidado (ambos meses) ────────────────────────────────────
df_ppto_final = pd.concat([df_ppto_08, df_ppto_09], ignore_index=True)
df_ppto_final['rubro'] = df_ppto_final['rubro'].str.strip()

ruta_ppto = PROC_DIR / 'presupuesto_clean.csv'
df_ppto_final.to_csv(ruta_ppto, index=False)
print(f'✅ presupuesto_clean.csv → {len(df_ppto_final)} registros')

# ── Tabla de mapeo de categorías ─────────────────────────────────────────────
df_mapeo = pd.DataFrame(
    list(MAPEO_CATEGORIAS.items()),
    columns=['categoria_gasto', 'rubro_presupuesto']
)
ruta_mapeo = PROC_DIR / 'mapeo_categorias.csv'
df_mapeo.to_csv(ruta_mapeo, index=False)
print(f'✅ mapeo_categorias.csv → {len(df_mapeo)} reglas de mapeo')

print()
print('RESUMEN FINAL')
print('─' * 50)
print(f'Gastos por miembro:')
print(df_gastos_final.groupby(['miembro', 'mes_origen'])['valor'].agg(['count','sum'])
      .rename(columns={'count':'registros','sum':'total_$'})
      .to_string())

✅ gastos_clean.csv  → 372 registros
✅ presupuesto_clean.csv → 74 registros
✅ mapeo_categorias.csv → 35 reglas de mapeo

RESUMEN FINAL
──────────────────────────────────────────────────
Gastos por miembro:
                    registros       total_$
miembro mes_origen                         
hijo    2023-09            56  5,677,360.00
mama    2023-08            27 50,425,697.00
        2023-09            46 12,207,858.27
papa    2023-08           117 61,887,897.84
        2023-09           126 25,187,627.84


In [11]:
# ── Verificación final ────────────────────────────────────────────────────────
print('VERIFICACIÓN DE ARCHIVOS EXPORTADOS')
print('─' * 50)
for f in PROC_DIR.iterdir():
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>6.1f} KB')

print()
print('✅ Notebook 01 completado — continuar con 02_data_model.ipynb')

VERIFICACIÓN DE ARCHIVOS EXPORTADOS
──────────────────────────────────────────────────
  gastos_clean.csv                           30.5 KB
  mapeo_categorias.csv                        0.9 KB
  presupuesto_clean.csv                       2.2 KB

✅ Notebook 01 completado — continuar con 02_data_model.ipynb
